In [1]:
#Importing libraries
import os
import json
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

### Building Tools

In [2]:
# Tool 1: Search Documents
knowledge_base = [
    {
        "topic": "RAG",
        "text": "Retrieval-Augmented Generation combines information retrieval with language generation. It retrieves relevant documents and provides them to an LLM as context."
    },
    {
        "topic": "Embeddings",
        "text": "Embeddings convert text into numerical vectors. Similar meanings can have vectors that are close together in vector space."
    },
    {
        "topic": "Vector databases",
        "text": "Vector databases store embeddings and support similarity searches to retrieve relevant information."
    },
    {
        "topic": "Chunking",
        "text": "Chunking splits large documents into smaller pieces. This helps retrieval systems find relevant information more precisely."
    }
]

def search_documents(query):
    query_words = set(query.lower().split())
    results = []

    for document in knowledge_base:
        text = document["text"].lower()
        score = sum(word in text for word in query_words)

        if score > 0:
            results.append({
                "topic": document["topic"],
                "text": document["text"],
                "score": score
            })

    results.sort(key=lambda item: item["score"], reverse=True)
    return results[:3]

In [4]:
#Tool 2 : Extract Key Points -- This function takes the search results and turns them into a list of points
\
def extract_key_points(search_results):
    points = []

    for result in search_results:
        points.append(
            result["topic"] + ": " + result["text"]
        )

    return points

In [5]:
#Tool 3 - Create a Summary
def create_summary(points):
    prompt = f"""
    Write a short, beginner-friendly summary
    using only the following information:

    {json.dumps(points, indent=2)}

    Do not add unsupported facts.
    """

    response = client.responses.create(
        model="gpt-4.1-mini",
        input=prompt
    )

    return response.output_text

In [6]:
#State
agent_state = {
    "goal": "Research RAG and summarize it",
    "search_results": [],
    "key_points": [],
    "final_summary": ""
}

In [8]:
#Multi step workflow -- Connecting all three tools
def run_research_workflow(goal):
    state = {
        "goal": goal,
        "search_results": [],
        "key_points": [],
        "final_summary": ""
    }

    # STEP 1: Search
    print("Step 1: Searching documents...")

    state["search_results"] = search_documents(goal)

    if not state["search_results"]:
        return {
            "status": "no_results",
            "state": state
        }

    # STEP 2: Extract key points
    print("Step 2: Extracting key points...")

    state["key_points"] = extract_key_points(
        state["search_results"]
    )

    # STEP 3: Generate summary
    print("Step 3: Generating summary...")

    state["final_summary"] = create_summary(
        state["key_points"]
    )

    return {
        "status": "completed",
        "state": state
    }

In [9]:
#Running workflow
result = run_research_workflow(
    "Explain RAG, embeddings, and vector databases"
)

print("Status:", result["status"])
print("\nFinal summary:")
print(result["state"]["final_summary"])

Step 1: Searching documents...
Step 2: Extracting key points...
Step 3: Generating summary...
Status: completed

Final summary:
Vector databases store embeddings, which are numerical vectors that represent text. These embeddings help find similar information by searching for vectors that are close together. Retrieval-Augmented Generation (RAG) uses this idea by finding relevant documents through retrieval and giving them to a language model to help generate better responses.


In [10]:
print("Search results:")
for item in result["state"]["search_results"]:
    print(item["topic"], "-", item["score"])

print("\nKey points:")
for point in result["state"]["key_points"]:
    print("-", point)

Search results:
Vector databases - 3
RAG - 1
Embeddings - 1

Key points:
- Vector databases: Vector databases store embeddings and support similarity searches to retrieve relevant information.
- RAG: Retrieval-Augmented Generation combines information retrieval with language generation. It retrieves relevant documents and provides them to an LLM as context.
- Embeddings: Embeddings convert text into numerical vectors. Similar meanings can have vectors that are close together in vector space.
